In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/adisongoh/it-service-ticket-classification-dataset/all_tickets_processed_improved_v3.csv


In [2]:
import pandas as pd
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# ==========================================
# 1. DEFINE PREPROCESSING FUNCTION FIRST
# ==========================================
def clean_ticket_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Strip trailing and leading whitespaces
    return text.strip()

# ==========================================
# 2. LOAD THE REAL-WORLD IT DATASET
# ==========================================
data_path = "/kaggle/input/datasets/adisongoh/it-service-ticket-classification-dataset/all_tickets_processed_improved_v3.csv"
df_it = pd.read_csv(data_path)
print(f"Successfully Loaded Real IT Dataset! Base Shape: {df_it.shape}")

# Sample 15,000 rows to optimize compilation speed and RAM usage
df_sample = df_it.sample(n=15000, random_state=42)

# ==========================================
# 3. APPLY NLP PREPROCESSING PIPELINE
# ==========================================
print("Cleaning unstructured ticket documents...")
df_sample['Cleaned_Document'] = df_sample['Document'].fillna('').apply(clean_ticket_text)

X_it = df_sample['Cleaned_Document']
y_it = df_sample['Topic_group']

# ==========================================
# 4. CONSTRUCT TRAIN-TEST SPLITS (80/20)
# ==========================================
X_train_it, X_test_it, y_train_it, y_test_it = train_test_split(X_it, y_it, test_size=0.2, random_state=42)

# ==========================================
# 5. MATHEMATICAL FEATURE EXTRACTION (TF-IDF)
# ==========================================
# Captures both individual terms and local phrase patterns (bi-grams)
tfidf_it = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf_it = tfidf_it.fit_transform(X_train_it)
X_test_tfidf_it = tfidf_it.transform(X_test_it)

# ==========================================
# 6. TRAIN THE SUPERVISED CLASSIFIER
# ==========================================
print("Training Support Vector Machine Classifier...")
model_it = LinearSVC(class_weight='balanced', random_state=42, max_iter=3000)
model_it.fit(X_train_tfidf_it, y_train_it)

# ==========================================
# 7. GENERATE SYSTEM OPERATIONAL METRICS
# ==========================================
preds_it = model_it.predict(X_test_tfidf_it)
print("\n================ SYSTEM PERFORMANCE REPORT ================")
print(classification_report(y_test_it, preds_it))

# ==========================================
# 8. SERIALIZE & EXPORT PRODUCTION COMPONENTS
# ==========================================
print("\nSerializing structural assets for local dashboard deployment...")
joblib.dump(model_it, 'it_ticket_classifier_model.pkl')
joblib.dump(tfidf_it, 'it_tfidf_vectorizer.pkl')
print("Assets successfully generated in /kaggle/working/!")

Successfully Loaded Real IT Dataset! Base Shape: (47837, 2)
Cleaning unstructured ticket documents...
Training Support Vector Machine Classifier...

================ SYSTEM PERFORMANCE REPORT ================
                       precision    recall  f1-score   support

               Access       0.90      0.85      0.87       452
Administrative rights       0.72      0.77      0.74       111
           HR Support       0.85      0.83      0.84       684
             Hardware       0.82      0.82      0.82       860
     Internal Project       0.80      0.80      0.80       132
        Miscellaneous       0.76      0.78      0.77       446
             Purchase       0.90      0.94      0.92       146
              Storage       0.82      0.87      0.84       169

             accuracy                           0.83      3000
            macro avg       0.82      0.83      0.83      3000
         weighted avg       0.83      0.83      0.83      3000


Serializing structural assets f